# Filter-bank CSP with LDA

Run the loading and QC cells to check the trial counts for your recordings. The saved outputs come from earlier exploratory runs on data that informed model development.

The saved full evaluation ended with a keyboard interrupt.

In [ ]:
from pathlib import Path
import sys

import mne
import numpy as np
import pandas as pd

from mne.decoding import CSP
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut
from sklearn.pipeline import make_pipeline

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "clean.py").is_file()),
)
sys.path.insert(0, str(PROJECT_ROOT))
from loader import get_files
from clean import load_dataset, save_qc

mne.set_log_level("WARNING")

### Load screened imagery trials

`load_dataset()` applies the fixed channel and trial QC rules in `clean.py`. The trial QC table keeps one row per original trial, including rejection reasons. This cell also updates the reports in `data/processed/qc`.

In [ ]:
files = get_files()
dataset = load_dataset(files)
epochs = dataset.epochs
qc_summary = save_qc(dataset, files)

print(f"Kept {len(epochs)} of {len(dataset.trial_qc)} trials")
print(f"Interpolated channel/run pairs: {dataset.channel_qc['interpolated'].sum()}")
print(qc_summary.loc[qc_summary["review_required"]].to_string(index=False))

### Prepare model inputs

`X` contains screened EEG in volts, with dimensions for trials, channels, and time samples. `y` contains the integer labels 1 and 2. `subjects` and `runs` identify the groups used to keep validation trials separate from training trials.

In [ ]:
X = epochs.get_data(copy=True)
y = epochs.events[:, -1]

subjects = epochs.metadata["subject"].to_numpy()
runs = epochs.metadata["run"].to_numpy()

print("EEG array:", X.shape)
print("Labels:", np.unique(y, return_counts=True))

### Check class balance

The table below reports left-hand and right-hand trial counts after QC for each subject. The analysis does not oversample either class. Review the flagged run and condition rows from the loading cell before interpreting model results.

In [ ]:
trial_table = epochs.metadata.copy()
trial_table["condition"] = np.where(
    y == 1,
    "left_hand",
    "right_hand",
)

pd.crosstab(
    trial_table["subject"],
    trial_table["condition"],
    margins=True,
)

## Evaluate filter-bank CSP with LDA

#### Tune filter-bank CSP components

`X_bank` has shape trials × bands × channels × samples. Each band has its own CSP, with a shared component count chosen from `[2, 4, 6]`. The small `FilterBankCSP` transformer keeps that choice in one grid parameter; separate pipeline branches would require linked parameter names.

Inner leave-one-subject-out CV chooses the component count using only the outer training subjects. The selected CSP models and shrinkage LDA are refitted on those subjects before predicting the outer held-out subject. `n_components` in the results means components per band, giving 6, 12, or 18 total features.

This cell evaluates the tuned filter-bank model only. Compare its per-subject scores with your separately tuned single-band CSP on the same trials and subjects. Band filtering is still applied to short epochs for this notebook experiment; continuous-recording filtering remains preferable for the final analysis.

Verification: the exact code cell passed a nested-CV smoke test through MNE using three archived subjects, eight channels, all three bands, and all three component candidates. Estimator cloning, feature dimensions, parameter extraction, and predictions passed. The full 64-channel evaluation was not rerun. Independent methods review passed; prior inspection of outer scores keeps model comparisons exploratory.


In [ ]:
class FilterBankCSP(TransformerMixin, BaseEstimator):
    def __init__(self, n_components=4):
        self.n_components = n_components

    def fit(self, X, y):
        self.csps_ = [
            CSP(
                n_components=self.n_components,
                reg="ledoit_wolf",
                log=True,
                norm_trace=False,
            ).fit(X[:, band], y)
            for band in range(X.shape[1])
        ]
        return self

    def transform(self, X):
        return np.concatenate([
            csp.transform(X[:, band])
            for band, csp in enumerate(self.csps_)
        ], axis=1)


bands = [(8, 12), (12, 20), (20, 30)]
X_bank = np.stack([
    mne.filter.filter_data(
        X,
        sfreq=epochs.info["sfreq"],
        l_freq=low,
        h_freq=high,
        copy=True,
        verbose=False,
    )
    for low, high in bands
], axis=1)

bank_pipeline = make_pipeline(
    FilterBankCSP(),
    SelectKBest(score_func=mutual_info_classif), # Jointly tunes features across bands
    LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"),
)

# Add it to your param_grid
param_grid = {
    "filterbankcsp__n_components": [2, 4, 6],
    "selectkbest__k": [4, 6] # Optimize how many total features to keep
}

comparison_rows = []

for train, test in LeaveOneGroupOut().split(X_bank, y, groups=subjects):
    search = GridSearchCV(
        bank_pipeline,
        param_grid,
        cv=LeaveOneGroupOut(),
        scoring="balanced_accuracy",
        refit=True,
        error_score="raise",
    )
    search.fit(X_bank[train], y[train], groups=subjects[train])
    bank_predictions = search.predict(X_bank[test])

    comparison_rows.append({
        "subject": int(subjects[test][0]),
        "n_components": search.best_params_["filterbankcsp__n_components"],
        "filter_bank_csp": balanced_accuracy_score(y[test], bank_predictions),
    })

comparison_results = pd.DataFrame(comparison_rows)
display(comparison_results)
print(
    "Mean balanced accuracy:",
    comparison_results["filter_bank_csp"].mean(),
)
